# Deep Learning para pronostico economico: Indicadores departamentales de Colombia

Los indicadores economicos de Colombia -- desempleo, inflacion, produccion industrial -- tienen patrones estacionales complejos y choques externos como el COVID que los modelos tradicionales no manejan bien. El desempleo sube cada enero, la inflacion tuvo un pico historico en 2022, y el COVID creo un choque que tomo 18 meses en recuperarse. Todo eso es no lineal, y ahi es donde entra deep learning.

Lo que vamos a hacer en este notebook es construir y comparar 4 arquitecturas de redes neuronales recurrentes para pronostico de series temporales:

1. **LSTM simple** -- la linea base. Las redes LSTM tienen "memoria": pueden aprender dependencias a largo plazo en series temporales.
2. **LSTM apilado** -- mas capas, mas capacidad para capturar patrones complejos.
3. **GRU** -- una alternativa mas ligera al LSTM con menos parametros.
4. **LSTM bidireccional** -- procesa la secuencia en ambas direcciones para capturar mejor el contexto.

La razon por la que comparo 4 arquitecturas es practica: no siempre necesitas el modelo mas complejo. A veces un GRU simple supera a un LSTM apilado porque el dataset es pequeno. Comparar nos dice que nivel de complejidad realmente necesitamos.

Al final obtenemos las mejores predicciones con forecasting multi-paso y analisis a nivel departamental.

## Configuracion del entorno

Lo primero es importar las librerias que necesitamos. Usamos TensorFlow/Keras para los modelos de deep learning y scikit-learn para el preprocesamiento. Tambien verificamos si hay GPU disponible -- los modelos LSTM entrenan significativamente mas rapido con GPU.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (14, 5), "font.size": 12})

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## Preparacion de datos -- Series temporales economicas de Colombia

Vamos a construir series temporales mensuales para 5 indicadores a nivel nacional y desempleo departamental para los 10 departamentos mas grandes de Colombia. El periodo es 2015-2024 (120 meses).

Lo que hay que esperar: las series incorporan patrones realistas -- ciclos estacionales de desempleo (mas alto en Q1), el choque del COVID en 2020, la crisis inflacionaria de 2022 y la dinamica de recuperacion post-pandemia.

La razon por la que uso datos con estas caracteristicas es que son exactamente el tipo de series que los modelos LSTM deberian poder capturar: tendencias de largo plazo, estacionalidad y choques estructurales.

In [ ]:
np.random.seed(42)

dates = pd.date_range(start="2015-01-01", periods=120, freq="MS")
months = np.arange(120)

# --- National Unemployment Rate (%) ---
# Base ~10%, seasonal pattern (higher in Q1), COVID spike mid-2020
unemployment_base = 10.0
unemployment_seasonal = 1.2 * np.sin(2 * np.pi * months / 12 - np.pi / 3)  # peaks ~Jan/Apr
unemployment_trend = -0.015 * months  # slow improvement pre-COVID

# COVID shock: sharp spike months 63-71 (Apr-Dec 2020), peak ~20% in May-Jul 2020
covid_shock_unemp = np.zeros(120)
for i in range(63, 72):
    covid_shock_unemp[i] = 10.0 * np.exp(-0.5 * ((i - 65) / 2.0) ** 2)
# Post-COVID gradual recovery
recovery_unemp = np.zeros(120)
for i in range(72, 120):
    recovery_unemp[i] = -min(3.0, 0.08 * (i - 72))

unemployment_noise = np.random.normal(0, 0.4, 120)
unemployment = (unemployment_base + unemployment_seasonal + unemployment_trend
                + covid_shock_unemp + recovery_unemp + unemployment_noise)
unemployment = np.clip(unemployment, 6.0, 22.0)

# --- Inflation Rate (%) ---
# ~3.5% baseline 2015-2019, drop in 2020, spike to ~13% in 2022, decline to ~7% by 2024
inflation_base = np.piecewise(
    months.astype(float),
    [months < 60, (months >= 60) & (months < 72), (months >= 72) & (months < 96), months >= 96],
    [lambda x: 3.5 + 0.01 * x,
     lambda x: 2.0 + 0.02 * (x - 60),
     lambda x: 2.5 + 0.14 * (x - 72),  # ramp up to ~13% by month 96
     lambda x: 13.0 - 0.25 * (x - 96)]  # decline from 13%
)
inflation_seasonal = 0.3 * np.sin(2 * np.pi * months / 12)
inflation_noise = np.random.normal(0, 0.25, 120)
inflation = inflation_base + inflation_seasonal + inflation_noise
inflation = np.clip(inflation, 1.5, 14.0)

# --- GDP Growth (%, quarterly interpolated to monthly) ---
gdp_quarterly = np.array([
    3.1, 3.0, 3.2, 2.8,  # 2015
    2.1, 2.3, 1.8, 1.6,  # 2016
    1.5, 1.2, 1.8, 1.4,  # 2017
    2.5, 2.8, 2.7, 2.6,  # 2018
    3.2, 3.0, 3.3, 3.4,  # 2019
    0.4, -15.8, -9.0, -3.6,  # 2020 (COVID)
    1.1, 17.6, 13.2, 10.7,  # 2021 (recovery/base effect)
    8.1, 12.6, 7.1, 2.9,  # 2022
    3.0, 1.6, -0.3, 0.3,  # 2023
    0.7, 1.8, 2.1, 1.9   # 2024
])
gdp_monthly = np.interp(months, np.linspace(0, 119, 40), gdp_quarterly)
gdp_monthly += np.random.normal(0, 0.3, 120)

# --- Consumer Confidence Index (CCI) ---
# Range roughly -30 to +10, drops sharply in 2020 and 2022
cci_base = np.piecewise(
    months.astype(float),
    [months < 60, (months >= 60) & (months < 72), (months >= 72) & (months < 90), months >= 90],
    [lambda x: -5 + 0.05 * x,
     lambda x: -30 + 0.8 * (x - 60),
     lambda x: -20 + 0.15 * (x - 72),
     lambda x: -15 + 0.1 * (x - 90)]
)
cci_seasonal = 2.0 * np.sin(2 * np.pi * months / 12 + np.pi / 4)
cci_noise = np.random.normal(0, 2.0, 120)
cci = cci_base + cci_seasonal + cci_noise

# --- Industrial Production Index (IPI, base 2015=100) ---
ipi_trend = 100 + 0.2 * months
ipi_covid = np.zeros(120)
for i in range(63, 72):
    ipi_covid[i] = -25.0 * np.exp(-0.5 * ((i - 65) / 2.5) ** 2)
ipi_recovery = np.zeros(120)
for i in range(72, 120):
    ipi_recovery[i] = min(5.0, 0.1 * (i - 72))
ipi_seasonal = 3.0 * np.sin(2 * np.pi * months / 12)
ipi_noise = np.random.normal(0, 1.5, 120)
ipi = ipi_trend + ipi_covid + ipi_recovery + ipi_seasonal + ipi_noise

# --- Assemble national DataFrame ---
df_national = pd.DataFrame({
    "date": dates,
    "unemployment": unemployment,
    "inflation": inflation,
    "gdp_growth": gdp_monthly,
    "consumer_confidence": cci,
    "industrial_production": ipi
}).set_index("date")

print(f"National indicators shape: {df_national.shape}")
df_national.describe().round(2)

In [ ]:
# --- Department-Level Unemployment (10 departments) ---
dept_config = {
    "Bogota":      {"base": 11.0, "amplitude": 1.0, "covid_peak": 12.0},
    "Antioquia":   {"base": 10.5, "amplitude": 1.1, "covid_peak": 10.0},
    "Valle":       {"base": 12.0, "amplitude": 1.3, "covid_peak": 11.0},
    "Santander":   {"base": 8.0,  "amplitude": 0.8, "covid_peak": 8.0},
    "Cundinamarca":{"base": 9.5,  "amplitude": 0.9, "covid_peak": 9.0},
    "Atlantico":   {"base": 9.0,  "amplitude": 1.0, "covid_peak": 9.5},
    "Bolivar":     {"base": 8.5,  "amplitude": 0.7, "covid_peak": 8.0},
    "Quindio":     {"base": 15.0, "amplitude": 1.5, "covid_peak": 11.0},
    "Choco":       {"base": 16.0, "amplitude": 1.2, "covid_peak": 9.0},
    "Narino":      {"base": 10.0, "amplitude": 1.0, "covid_peak": 8.5},
}

df_dept = pd.DataFrame({"date": dates}).set_index("date")

for dept, cfg in dept_config.items():
    base = cfg["base"]
    amp = cfg["amplitude"]
    cpk = cfg["covid_peak"]

    seasonal = amp * np.sin(2 * np.pi * months / 12 - np.pi / 3)
    trend = -0.01 * months

    covid = np.zeros(120)
    for i in range(63, 72):
        covid[i] = cpk * np.exp(-0.5 * ((i - 65) / 2.0) ** 2)

    recovery = np.zeros(120)
    for i in range(72, 120):
        recovery[i] = -min(2.5, 0.06 * (i - 72))

    noise = np.random.normal(0, 0.5, 120)
    series = base + seasonal + trend + covid + recovery + noise
    df_dept[dept] = np.clip(series, 4.0, 28.0)

print(f"Department unemployment shape: {df_dept.shape}")
df_dept.head()

In [ ]:
# --- Visualize national indicators ---
fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)

indicators = ["unemployment", "inflation", "gdp_growth", "consumer_confidence", "industrial_production"]
titles = ["Unemployment Rate (%)", "Inflation Rate (%)", "GDP Growth (%)",
          "Consumer Confidence Index", "Industrial Production Index"]
colors = ["#e74c3c", "#f39c12", "#2ecc71", "#3498db", "#9b59b6"]

for idx, (col, title, color) in enumerate(zip(indicators, titles, colors)):
    ax = axes.flat[idx]
    ax.plot(df_national.index, df_national[col], color=color, linewidth=1.5)
    ax.axvline(pd.Timestamp("2020-03-01"), color="gray", linestyle="--", alpha=0.6, label="COVID-19")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylabel(title.split("(")[-1].replace(")", "").strip() if "(" in title else "Value")
    if idx == 0:
        ax.legend(fontsize=10)

axes.flat[-1].axis("off")
plt.suptitle("Colombian National Economic Indicators (2015-2024)", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Visualize department-level unemployment ---
fig, ax = plt.subplots(figsize=(16, 6))
for dept in df_dept.columns:
    ax.plot(df_dept.index, df_dept[dept], linewidth=1.2, label=dept)
ax.axvline(pd.Timestamp("2020-03-01"), color="gray", linestyle="--", alpha=0.6)
ax.set_title("Department-Level Unemployment Rates (2015-2024)", fontsize=14, fontweight="bold")
ax.set_ylabel("Unemployment Rate (%)")
ax.set_xlabel("Date")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

## Ingenieria de features -- Crear variables que ayuden al modelo a aprender patrones

Aqui es donde enriquecemos los datos crudos con features que le facilitan la vida al modelo. Lo que hacemos es crear:

- **Lags** (1, 3, 6, 12 meses): el valor del desempleo hace N meses. La razon por la que uso lags es que le dan al modelo acceso explicito a valores pasados.
- **Estadisticas rodantes** (media movil, desviacion estandar): capturan la tendencia reciente y la volatilidad.
- **Codificacion ciclica del tiempo** (seno/coseno del mes y trimestre): la razon por la que codifico el mes como seno/coseno en vez de un numero 1-12 es que asi el modelo entiende que diciembre y enero estan cerca, no lejos.

Despues normalizamos todo al rango [0, 1] con MinMaxScaler y creamos secuencias deslizantes de 12 meses. Cada secuencia es una ventana de 12 meses que el modelo usa como input para predecir el siguiente mes.

In [ ]:
# Target: national unemployment rate
TARGET_COL = "unemployment"

df_feat = df_national.copy()

# Lag features
for lag in [1, 3, 6, 12]:
    df_feat[f"{TARGET_COL}_lag_{lag}"] = df_feat[TARGET_COL].shift(lag)

# Rolling statistics
for window in [3, 6, 12]:
    df_feat[f"{TARGET_COL}_ma_{window}"] = df_feat[TARGET_COL].rolling(window).mean()
    df_feat[f"{TARGET_COL}_std_{window}"] = df_feat[TARGET_COL].rolling(window).std()

# Cyclical month/quarter encoding
df_feat["month_sin"] = np.sin(2 * np.pi * df_feat.index.month / 12)
df_feat["month_cos"] = np.cos(2 * np.pi * df_feat.index.month / 12)
df_feat["quarter_sin"] = np.sin(2 * np.pi * df_feat.index.quarter / 4)
df_feat["quarter_cos"] = np.cos(2 * np.pi * df_feat.index.quarter / 4)

# Drop rows with NaN from lagging/rolling
df_feat = df_feat.dropna()
print(f"Feature matrix shape after engineering: {df_feat.shape}")
print(f"Features: {list(df_feat.columns)}")

In [ ]:
# Normalize all features to [0, 1]
feature_cols = [c for c in df_feat.columns if c != TARGET_COL]
all_cols = [TARGET_COL] + feature_cols

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_feat[all_cols])
df_scaled = pd.DataFrame(scaled_data, columns=all_cols, index=df_feat.index)

# We need a separate scaler for the target to inverse-transform predictions later
target_scaler = MinMaxScaler()
target_scaler.fit(df_feat[[TARGET_COL]])

print(f"Scaled data shape: {df_scaled.shape}")
df_scaled.head()

In [ ]:
# Create sliding-window sequences for LSTM input
WINDOW_SIZE = 12  # 12 months lookback

def create_sequences(data, target_col_idx, window_size, forecast_horizon=1):
    """Create (X, y) pairs from scaled time series.
    X shape: (samples, window_size, n_features)
    y shape: (samples,) for horizon=1 or (samples, horizon) for multi-step
    """
    X, y = [], []
    for i in range(window_size, len(data) - forecast_horizon + 1):
        X.append(data[i - window_size:i, :])
        if forecast_horizon == 1:
            y.append(data[i, target_col_idx])
        else:
            y.append(data[i:i + forecast_horizon, target_col_idx])
    return np.array(X), np.array(y)

scaled_values = df_scaled.values
target_col_idx = 0  # unemployment is first column

X_all, y_all = create_sequences(scaled_values, target_col_idx, WINDOW_SIZE, forecast_horizon=1)
print(f"Sequences X shape: {X_all.shape}")
print(f"Sequences y shape: {y_all.shape}")
print(f"Number of features per timestep: {X_all.shape[2]}")

## Division train/test

Algo fundamental en series temporales: nunca se mezclan los datos aleatoriamente. Usamos una division temporal -- entrenamos con 2015-2022 y testeamos con 2023-2024.

La razon por la que no usamos cross-validation tradicional aqui es que las series temporales tienen un orden secuencial. Si mezclaramos datos de 2024 en el entrenamiento, el modelo estaria "viendo el futuro", lo cual invalida completamente la evaluacion.

In [ ]:
# The sequences correspond to dates starting at index WINDOW_SIZE of df_scaled
seq_dates = df_scaled.index[WINDOW_SIZE:]

# Find split point: first date in 2023
split_idx = np.argmax(seq_dates >= "2023-01-01")
print(f"Split index: {split_idx} | Train samples: {split_idx} | Test samples: {len(seq_dates) - split_idx}")

X_train, X_test = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]
dates_train, dates_test = seq_dates[:split_idx], seq_dates[split_idx:]

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")
print(f"Train period: {dates_train[0].strftime('%Y-%m')} to {dates_train[-1].strftime('%Y-%m')}")
print(f"Test period:  {dates_test[0].strftime('%Y-%m')} to {dates_test[-1].strftime('%Y-%m')}")

In [ ]:
# Visualize train/test split on the raw unemployment series
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dates_train, target_scaler.inverse_transform(y_train.reshape(-1, 1)).flatten(),
        color="#2980b9", linewidth=1.5, label="Train")
ax.plot(dates_test, target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten(),
        color="#e74c3c", linewidth=1.5, label="Test")
ax.axvline(pd.Timestamp("2023-01-01"), color="black", linestyle="--", alpha=0.7, label="Split")
ax.set_title("Train / Test Split -- National Unemployment Rate", fontsize=14, fontweight="bold")
ax.set_ylabel("Unemployment Rate (%)")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Shared training utilities ---

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error."""
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def get_callbacks():
    """Standard callbacks for all models."""
    return [
        EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-6, verbose=0)
    ]

def evaluate_model(model, X_test, y_test, target_scaler, model_name):
    """Predict, inverse-transform, and compute metrics."""
    y_pred_scaled = model.predict(X_test, verbose=0).flatten()
    y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_actual = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
    mae = mean_absolute_error(y_actual, y_pred)
    mape_val = mape(y_actual, y_pred)

    print(f"[{model_name}]  RMSE: {rmse:.4f}  |  MAE: {mae:.4f}  |  MAPE: {mape_val:.2f}%")
    return y_pred, y_actual, {"model": model_name, "rmse": rmse, "mae": mae, "mape": mape_val}

# Store results for comparison
results = []
predictions = {}

N_FEATURES = X_train.shape[2]
print(f"Input shape for models: ({WINDOW_SIZE}, {N_FEATURES})")

## Modelo 1: LSTM simple -- La linea base de deep learning

Nuestro primer modelo es un LSTM con una sola capa de 64 unidades, seguido de Dropout para regularizacion y dos capas Dense.

La razon por la que empezamos con esto es que es nuestra linea base. Todo lo que viene despues (mas capas, arquitecturas diferentes) tiene que superar a este modelo para justificar su complejidad adicional.

Lo que hay que esperar: un modelo relativamente rapido de entrenar que ya deberia capturar los patrones estacionales basicos del desempleo.

In [ ]:
tf.random.set_seed(42)

model_simple_lstm = Sequential([
    LSTM(64, input_shape=(WINDOW_SIZE, N_FEATURES)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
], name="Simple_LSTM")

model_simple_lstm.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
model_simple_lstm.summary()

In [ ]:
t0 = time.time()
history_simple = model_simple_lstm.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    callbacks=get_callbacks(),
    verbose=0
)
train_time_simple = time.time() - t0
print(f"Training completed in {train_time_simple:.1f}s | Epochs run: {len(history_simple.history['loss'])}")

In [ ]:
# Training curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_simple.history["loss"], label="Train Loss", linewidth=1.5)
ax.plot(history_simple.history["val_loss"], label="Validation Loss", linewidth=1.5)
ax.set_title("Simple LSTM -- Training and Validation Loss", fontsize=13, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.legend()
plt.tight_layout()
plt.show()

# Evaluate
y_pred_simple, y_actual, metrics_simple = evaluate_model(
    model_simple_lstm, X_test, y_test, target_scaler, "Simple LSTM"
)
metrics_simple["train_time"] = train_time_simple
results.append(metrics_simple)
predictions["Simple LSTM"] = y_pred_simple

## Modelo 2: LSTM apilado -- Mas capas, mas capacidad

Ahora apilamos dos capas LSTM. La primera capa tiene `return_sequences=True`, lo que significa que pasa la secuencia completa de estados ocultos a la segunda capa en vez de solo el ultimo.

La razon por la que esto puede mejorar los resultados es que la segunda capa aprende representaciones de nivel superior sobre los patrones que la primera ya extrajo. Es como tener dos niveles de abstraccion.

Lo que hay que esperar: mas parametros, entrenamiento mas lento, y potencialmente mejor captura de patrones complejos. Tambien uso mas Dropout (0.3 entre capas) porque modelos mas grandes son mas propensos a sobreajustarse.

In [ ]:
tf.random.set_seed(42)

model_stacked_lstm = Sequential([
    LSTM(128, return_sequences=True, input_shape=(WINDOW_SIZE, N_FEATURES)),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
], name="Stacked_LSTM")

model_stacked_lstm.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
model_stacked_lstm.summary()

In [ ]:
t0 = time.time()
history_stacked = model_stacked_lstm.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    callbacks=get_callbacks(),
    verbose=0
)
train_time_stacked = time.time() - t0
print(f"Training completed in {train_time_stacked:.1f}s | Epochs run: {len(history_stacked.history['loss'])}")

In [ ]:
# Training curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_stacked.history["loss"], label="Train Loss", linewidth=1.5)
ax.plot(history_stacked.history["val_loss"], label="Validation Loss", linewidth=1.5)
ax.set_title("Stacked LSTM -- Training and Validation Loss", fontsize=13, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.legend()
plt.tight_layout()
plt.show()

# Evaluate
y_pred_stacked, _, metrics_stacked = evaluate_model(
    model_stacked_lstm, X_test, y_test, target_scaler, "Stacked LSTM"
)
metrics_stacked["train_time"] = train_time_stacked
results.append(metrics_stacked)
predictions["Stacked LSTM"] = y_pred_stacked

## Modelo 3: GRU -- Una alternativa mas ligera al LSTM

El GRU (Gated Recurrent Unit) fusiona las puertas de olvido e input del LSTM en una sola puerta de actualizacion. Resultado: menos parametros y entrenamiento tipicamente mas rapido.

La razon por la que lo incluyo es practica: en muchos problemas reales, el GRU iguala o supera al LSTM, especialmente cuando el dataset es pequeno (como el nuestro con aproximadamente 100 muestras de entrenamiento). Menos parametros significa menos riesgo de sobreajuste.

In [ ]:
tf.random.set_seed(42)

model_gru = Sequential([
    GRU(64, input_shape=(WINDOW_SIZE, N_FEATURES)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
], name="GRU")

model_gru.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
model_gru.summary()

In [ ]:
t0 = time.time()
history_gru = model_gru.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    callbacks=get_callbacks(),
    verbose=0
)
train_time_gru = time.time() - t0
print(f"Training completed in {train_time_gru:.1f}s | Epochs run: {len(history_gru.history['loss'])}")

# Evaluate
y_pred_gru, _, metrics_gru = evaluate_model(
    model_gru, X_test, y_test, target_scaler, "GRU"
)
metrics_gru["train_time"] = train_time_gru
results.append(metrics_gru)
predictions["GRU"] = y_pred_gru

## Modelo 4: LSTM bidireccional -- Ver el contexto en ambas direcciones

El wrapper Bidirectional procesa la secuencia de input en ambas direcciones (adelante y atras) y concatena las salidas.

Voy a ser honesta: para pronostico de series temporales estrictamente causal, esto es debatible -- en produccion no tenemos informacion del futuro. Pero en la practica, el pase hacia atras ayuda al modelo a aprender representaciones mas ricas de las secuencias de entrenamiento, especialmente cuando la ventana contiene rupturas estructurales como el COVID.

Lo que hay que esperar: el doble de parametros comparado con el LSTM simple (porque procesa en dos direcciones) y potencialmente mejores representaciones internas.

In [ ]:
tf.random.set_seed(42)

model_bilstm = Sequential([
    Bidirectional(LSTM(64), input_shape=(WINDOW_SIZE, N_FEATURES)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
], name="Bidirectional_LSTM")

model_bilstm.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
model_bilstm.summary()

In [ ]:
t0 = time.time()
history_bilstm = model_bilstm.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    callbacks=get_callbacks(),
    verbose=0
)
train_time_bilstm = time.time() - t0
print(f"Training completed in {train_time_bilstm:.1f}s | Epochs run: {len(history_bilstm.history['loss'])}")

# Evaluate
y_pred_bilstm, _, metrics_bilstm = evaluate_model(
    model_bilstm, X_test, y_test, target_scaler, "Bidirectional LSTM"
)
metrics_bilstm["train_time"] = train_time_bilstm
results.append(metrics_bilstm)
predictions["Bidirectional LSTM"] = y_pred_bilstm

## Forecasting multi-paso -- Predecir los proximos 3 meses

Hasta ahora todos los modelos predicen un solo mes adelante. Pero en la practica, necesitamos pronosticos de horizonte mas largo.

Lo que hacemos aqui es modificar la mejor arquitectura para que la capa de salida tenga 3 neuronas en vez de 1, una por cada mes del horizonte de pronostico. El loss se calcula como el promedio del error en los 3 pasos.

Lo que hay que esperar: la precision se degrada con cada paso adicional. El pronostico a t+1 siempre va a ser mejor que a t+3 porque la incertidumbre se acumula. La pregunta interesante es cuanto se degrada.

In [ ]:
# Create multi-step sequences (horizon=3)
HORIZON = 3
X_all_ms, y_all_ms = create_sequences(scaled_values, target_col_idx, WINDOW_SIZE, forecast_horizon=HORIZON)

# Split
split_idx_ms = np.argmax(df_scaled.index[WINDOW_SIZE:len(df_scaled) - HORIZON + 1] >= "2023-01-01")
X_train_ms, X_test_ms = X_all_ms[:split_idx_ms], X_all_ms[split_idx_ms:]
y_train_ms, y_test_ms = y_all_ms[:split_idx_ms], y_all_ms[split_idx_ms:]

print(f"Multi-step X_train: {X_train_ms.shape}, y_train: {y_train_ms.shape}")
print(f"Multi-step X_test:  {X_test_ms.shape},  y_test:  {y_test_ms.shape}")

In [ ]:
tf.random.set_seed(42)

model_multistep = Sequential([
    LSTM(128, return_sequences=True, input_shape=(WINDOW_SIZE, N_FEATURES)),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(HORIZON)  # 3-month output
], name="MultiStep_LSTM")

model_multistep.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
model_multistep.summary()

t0 = time.time()
history_ms = model_multistep.fit(
    X_train_ms, y_train_ms,
    epochs=100,
    batch_size=16,
    validation_split=0.15,
    callbacks=get_callbacks(),
    verbose=0
)
train_time_ms = time.time() - t0
print(f"\nTraining completed in {train_time_ms:.1f}s | Epochs run: {len(history_ms.history['loss'])}")

In [ ]:
# Evaluate multi-step: RMSE per horizon step
y_pred_ms_scaled = model_multistep.predict(X_test_ms, verbose=0)

print("Multi-Step Forecasting -- RMSE by Horizon:")
for step in range(HORIZON):
    y_true_step = target_scaler.inverse_transform(y_test_ms[:, step].reshape(-1, 1)).flatten()
    y_pred_step = target_scaler.inverse_transform(y_pred_ms_scaled[:, step].reshape(-1, 1)).flatten()
    rmse_step = np.sqrt(mean_squared_error(y_true_step, y_pred_step))
    mae_step = mean_absolute_error(y_true_step, y_pred_step)
    print(f"  t+{step+1} month:  RMSE = {rmse_step:.4f}  |  MAE = {mae_step:.4f}")

## Comparacion de modelos -- Cual arquitectura funciona mejor y por que

Ahora vamos a comparar los 4 modelos lado a lado. Lo que miramos es:
- **RMSE** (Root Mean Squared Error): penaliza mas los errores grandes
- **MAE** (Mean Absolute Error): el error promedio en las mismas unidades de la variable
- **MAPE** (Mean Absolute Percentage Error): el error como porcentaje, mas facil de interpretar

Tambien analizamos los residuales del mejor modelo para verificar que no haya patrones sistematicos en los errores. Si los residuales muestran un patron, significa que el modelo no esta capturando algo.

In [ ]:
# Comparison table
df_results = pd.DataFrame(results)
df_results["train_time"] = df_results["train_time"].map(lambda x: f"{x:.1f}s")
df_results = df_results.rename(columns={
    "model": "Model", "rmse": "RMSE", "mae": "MAE", "mape": "MAPE (%)", "train_time": "Train Time"
})
df_results = df_results.sort_values("RMSE").reset_index(drop=True)
print("Model Comparison (sorted by RMSE):\n")
df_results.style.format({"RMSE": "{:.4f}", "MAE": "{:.4f}", "MAPE (%)": "{:.2f}"})

In [ ]:
# Best model: actual vs predicted (full test period)
best_model_name = df_results.iloc[0]["Model"]
best_preds = predictions[best_model_name]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates_test, y_actual, color="#2c3e50", linewidth=2, label="Actual", marker="o", markersize=4)
ax.plot(dates_test, best_preds, color="#e74c3c", linewidth=2, label=f"Predicted ({best_model_name})",
        marker="s", markersize=4, linestyle="--")
ax.fill_between(dates_test, y_actual, best_preds, alpha=0.15, color="#e74c3c")
ax.set_title(f"Best Model: {best_model_name} -- Actual vs Predicted Unemployment",
             fontsize=14, fontweight="bold")
ax.set_ylabel("Unemployment Rate (%)")
ax.set_xlabel("Date")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# All models overlaid
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates_test, y_actual, color="#2c3e50", linewidth=2.5, label="Actual", zorder=5)
model_colors = {"Simple LSTM": "#e74c3c", "Stacked LSTM": "#3498db",
                "GRU": "#2ecc71", "Bidirectional LSTM": "#9b59b6"}
for name, preds in predictions.items():
    ax.plot(dates_test, preds, linewidth=1.5, linestyle="--", label=name,
            color=model_colors.get(name, "gray"), alpha=0.85)
ax.set_title("All Models -- Predicted vs Actual Unemployment (Test Period)",
             fontsize=14, fontweight="bold")
ax.set_ylabel("Unemployment Rate (%)")
ax.set_xlabel("Date")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Residual analysis for best model
residuals = y_actual - best_preds

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Residual distribution
axes[0].hist(residuals, bins=15, color="#3498db", edgecolor="white", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5)
axes[0].set_title("Residual Distribution", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Residual (Actual - Predicted)")
axes[0].set_ylabel("Frequency")

# Residuals over time
axes[1].plot(dates_test, residuals, color="#2c3e50", linewidth=1.5, marker="o", markersize=4)
axes[1].axhline(0, color="red", linestyle="--", linewidth=1.5)
axes[1].set_title("Residuals Over Time", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Residual")

# Autocorrelation of residuals (manual calculation)
max_lag = min(12, len(residuals) - 1)
autocorrs = [np.corrcoef(residuals[:-lag], residuals[lag:])[0, 1] if lag > 0 else 1.0
             for lag in range(max_lag + 1)]
axes[2].bar(range(max_lag + 1), autocorrs, color="#9b59b6", edgecolor="white")
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].axhline(1.96 / np.sqrt(len(residuals)), color="red", linestyle="--", alpha=0.6)
axes[2].axhline(-1.96 / np.sqrt(len(residuals)), color="red", linestyle="--", alpha=0.6)
axes[2].set_title("Residual Autocorrelation", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Lag")
axes[2].set_ylabel("Autocorrelation")

plt.suptitle(f"Residual Analysis -- {best_model_name}", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

## Analisis a nivel departamental

Hasta ahora trabajamos con datos nacionales. Ahora vamos a aplicar la mejor arquitectura a cada uno de los 10 departamentos de forma independiente, entrenando un modelo LSTM separado por departamento.

La razon por la que entreno modelos separados (en vez de un solo modelo multi-departamento) es que cada departamento tiene dinamicas propias. Quindio y Choco tienen tasas de desempleo base mucho mas altas que Bogota, y el impacto del COVID fue diferente en cada uno.

Lo que hay que esperar: los departamentos con mayor volatilidad (Quindio, Choco) probablemente tendran mayores errores de pronostico. El heatmap final resume el error por departamento y trimestre.

In [ ]:
def build_dept_model(n_features):
    """Build a lightweight LSTM for single-department forecasting."""
    model = Sequential([
        LSTM(32, input_shape=(WINDOW_SIZE, n_features)),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
    return model

dept_actual = {}
dept_predicted = {}
dept_rmse = {}

for dept in df_dept.columns:
    tf.random.set_seed(42)

    # Use department unemployment as single feature
    series = df_dept[[dept]].values
    scaler_d = MinMaxScaler()
    scaled_d = scaler_d.fit_transform(series)

    # Create sequences
    X_d, y_d = [], []
    for i in range(WINDOW_SIZE, len(scaled_d)):
        X_d.append(scaled_d[i - WINDOW_SIZE:i, :])
        y_d.append(scaled_d[i, 0])
    X_d, y_d = np.array(X_d), np.array(y_d)

    # Split at 2023
    d_dates = df_dept.index[WINDOW_SIZE:]
    d_split = np.argmax(d_dates >= "2023-01-01")
    X_tr_d, X_te_d = X_d[:d_split], X_d[d_split:]
    y_tr_d, y_te_d = y_d[:d_split], y_d[d_split:]

    # Train
    m = build_dept_model(1)
    m.fit(X_tr_d, y_tr_d, epochs=60, batch_size=8, validation_split=0.15,
          callbacks=get_callbacks(), verbose=0)

    # Predict
    y_p = m.predict(X_te_d, verbose=0).flatten()
    y_p_inv = scaler_d.inverse_transform(y_p.reshape(-1, 1)).flatten()
    y_a_inv = scaler_d.inverse_transform(y_te_d.reshape(-1, 1)).flatten()

    dept_actual[dept] = y_a_inv
    dept_predicted[dept] = y_p_inv
    dept_rmse[dept] = np.sqrt(mean_squared_error(y_a_inv, y_p_inv))

print("Department-level RMSE:")
for dept, rmse_val in sorted(dept_rmse.items(), key=lambda x: x[1]):
    print(f"  {dept:15s}: {rmse_val:.4f}")

In [ ]:
# Heatmap: absolute error (predicted - actual) by department and quarter
test_months_dept = df_dept.index[WINDOW_SIZE:][d_split:]

# Use average of test period split into quarters for a cleaner heatmap
n_test = len(list(dept_actual.values())[0])
quarter_labels = []
for i in range(0, n_test, 3):
    end = min(i + 3, n_test)
    label = test_months_dept[i].strftime("%Y-Q") + str((test_months_dept[i].month - 1) // 3 + 1)
    quarter_labels.append(label)

heatmap_data = pd.DataFrame(index=df_dept.columns)
for qi, i in enumerate(range(0, n_test, 3)):
    end = min(i + 3, n_test)
    col_label = quarter_labels[qi]
    for dept in df_dept.columns:
        heatmap_data.loc[dept, col_label] = np.mean(
            np.abs(dept_predicted[dept][i:end] - dept_actual[dept][i:end])
        )

heatmap_data = heatmap_data.astype(float)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="YlOrRd", linewidths=0.5,
            cbar_kws={"label": "Mean Absolute Error (pp)"}, ax=ax)
ax.set_title("Department-Level Forecast Error (MAE) by Quarter", fontsize=14, fontweight="bold")
ax.set_ylabel("Department")
ax.set_xlabel("Quarter")
plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs actual unemployment summary (average over test period)
comparison = pd.DataFrame({
    "Actual (avg)": {d: np.mean(dept_actual[d]) for d in df_dept.columns},
    "Predicted (avg)": {d: np.mean(dept_predicted[d]) for d in df_dept.columns},
    "RMSE": dept_rmse
}).round(3)
comparison = comparison.sort_values("Actual (avg)", ascending=False)
print("Department-Level Summary (Test Period Averages):\n")
comparison

## Hallazgos principales

In [ ]:
# Summary metrics
print("=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)
for _, row in df_results.iterrows():
    print(f"  {row['Model']:25s}  RMSE={row['RMSE']:.4f}  MAE={row['MAE']:.4f}  "
          f"MAPE={row['MAPE (%)']:.2f}%  Time={row['Train Time']}")
print("=" * 70)
print(f"\nBest single-step model: {df_results.iloc[0]['Model']}")
print(f"Department-level avg RMSE: {np.mean(list(dept_rmse.values())):.4f}")
print(f"Multi-step horizon degradation: t+1 to t+3 months")

### Resumen de resultados

**Comparacion de arquitecturas:**
- Las 4 arquitecturas recurrentes logran una precision razonable para pronostico de corto plazo del desempleo colombiano.
- El LSTM apilado y el GRU tienden a tener rendimiento comparable, con el GRU ofreciendo entrenamiento mas rapido gracias a su estructura de puertas mas simple.
- El LSTM bidireccional agrega poder representacional pero su valor es debatible para pronostico estrictamente causal.

**Lo que aprendi en la practica:**
- Los modelos LSTM capturan efectivamente los ciclos estacionales de desempleo (mas alto en Q1/Q4) y la trayectoria de recuperacion post-COVID -- patrones que modelos estadisticos mas simples (ARIMA, suavizamiento exponencial) requieren ajuste manual para manejar conjuntamente.
- El forecasting multi-paso confirma la degradacion esperada: el RMSE a t+3 es significativamente mayor que a t+1, reflejando la acumulacion de incertidumbre.
- Los modelos departamentales muestran que regiones con mayor desempleo base (Quindio, Choco) y choques COVID mas grandes tienden a tener mayores errores de pronostico, consistente con la mayor volatilidad en esas series.

**Limitaciones que hay que tener en cuenta:**
- 120 observaciones mensuales es un dataset pequeno para deep learning. En produccion, datos diarios o semanales y periodos mas largos mejorarian la generalizacion.
- Los choques externos (pandemia, cambios de politica, precios globales de commodities) son inherentemente impredecibles -- ningun modelo puramente autorregresivo puede pronosticarlos. Incorporar indicadores lider exogenos o sentimiento de noticias podria ayudar.
- La optimizacion de hiperparametros fue minima; una busqueda sistematica (Keras Tuner, Optuna) probablemente mejoraria los resultados.